# CSP-10-Optimisation-Z3-Rust : Couverture Minimale avec le Solveur SMT Z3

**Navigation** : [<< CSP-09-Distributed-Python](CSP-09-Distributed-Python.ipynb) | [Index CSP](README.md) | [Version C#](CSP-10-Optimisation-Z3-CSharp.ipynb) | [Version Python](CSP-10-Optimisation-Z3-Python.ipynb) | [Partie 3 : Cas pratiques >>](../Part3-Cas-pratiques/README.md)

> **Durée estimée** : 1h30

> **Solveur** : Z3 (crate Rust `z3` 0.24.0) — solveur SMT. Ce notebook est le **pont CP ↔ SMT** de la partie : le même problème d'optimisation combinatoire, formulé une seule fois, résolu par un solveur SMT au lieu de CP-SAT.

## La couverture minimale d'ensembles, vue par un solveur SMT

Ce notebook prend le problème de **couverture minimale d'ensembles** (*set cover*) — NP-complet, fondamental en placement de capteurs, en logistique et en théorie des réseaux — et le résout avec **Z3**, le solveur SMT de référence. L'objet pédagogique est double : montrer comment un problème CP classique se réécrit en SMT d'optimisation, et mesurer **où le solveur prend le dessus** sur une baseline brute-force.

## Objectifs

À la fin de ce notebook, vous saurez :
1. **Formuler** un problème de couverture minimale comme un problème SMT d'optimisation dans Z3
2. **Opposer** une baseline brute-force (énumération 2^k) au solveur, et identifier le point de bascule
3. **Lire** un optimum SMT et **en extraire la preuve de minimalité** (aucune couverture plus petite n'existe)
4. **Comprendre** le pont CP ↔ SAT ↔ SMT : même formulation déclarative, solveurs interchangeables

## Prérequis

- Notebooks CSP-1 à CSP-2 (le paradigme déclaratif : variables, contraintes, solveur)
- CSP-05-Optimization-Python (optimisation combinatoire sous CP-SAT)
- Rust stable sur le noyau `rust` (evcxr) : crate `z3` 0.24.0 ; le rendu de la grille est **textuel** (aucune figure n'est produite dans ce portage)

> *Ancres savantes -- de Moura, L. & Bjørner, N. (2008), Z3: An Efficient SMT Solver, TACAS 2008, LNCS 4956:357-371 (le solveur SMT utilisé ici, ses théories et son moteur d'optimisation) ; Garey, M.R. & Johnson, D.S. (1979), Computers and Intractability, W.H. Freeman (la couverture d'ensembles, set cover, parmi les 21 problèmes NP-complets fondateurs) ; Bardin, S., Bjørner, N. & Cadar, C. (2019), Bringing CP, SAT and SMT together, Dagstuhl Reports 9(2):27 (le pont entre les trois familles de solveurs que ce notebook franchit).*

In [2]:
:dep z3 = { version = "0.24.0", features = ["gh-release"] }


## 1. Le problème : la couverture minimale d'ensembles

La **couverture minimale d'ensembles** (*minimum set cover*) est l'un des problèmes NP-complets fondateurs. On donne :

- un **univers** $U$ d'éléments à couvrir (ici des points d'une grille) ;
- une **famille** $\mathcal{S}$ de sous-ensembles de $U$ (ici les zones couvertes par chaque capteur candidat).

L'objectif : choisir le **plus petit** sous-famille de $\mathcal{S}$ dont l'union recouvre tout $U$. C'est le problème abstrait derrière le placement de capteurs, l'allumage de relais, la sélection de tests diagnostiques, la couverture de plans de production.

Sa difficulté est exactement celle qui intéresse cette série : le problème est **NP-complet**, donc aucune baseline ne le résout en général à grande échelle — c'est le terrain où un solveur se mesure.

### Application : capteurs sur une grille

Concrétisons : une grille de $r \times c$ points, et **un capteur candidat par case**. Un capteur placé sur une case couvre le **carré 3×3** centré sur cette case (borné aux limites de la grille). La question : *quels capteurs allumer, en nombre minimal, pour que chaque point soit couvert par au moins un capteur ?*

C'est un cas particulier de l'**ensemble dominant** d'un graphe (graphe de grille), et la formulation en « points → capteurs qui le couvrent » est la forme d'entrée naturelle d'un solveur.

In [3]:
use std::collections::HashMap;

// Cellule autonome : la construction de grille est redéfinie ici,
// sous evcxr les fonctions ne sont pas persistées d'une cellule à l'autre.
fn build_grid_cover(rows: i64, cols: i64) -> (Vec<(i64, i64)>, HashMap<usize, Vec<usize>>, HashMap<(i64, i64), usize>) {
    // points : liste des (r, c) ; point_index : (r, c) -> index du point
    let mut points: Vec<(i64, i64)> = Vec::new();
    for r in 0..rows {
        for c in 0..cols {
            points.push((r, c));
        }
    }
    let mut point_index: HashMap<(i64, i64), usize> = HashMap::new();
    for (i, p) in points.iter().enumerate() {
        point_index.insert(*p, i);
    }
    // point_to_sensors : pour chaque point, les indices de capteurs qui le couvrent
    let mut point_to_sensors: HashMap<usize, Vec<usize>> = HashMap::new();
    for (sr, sc) in points.iter().copied() {
        let sidx = match point_index.get(&(sr, sc)) {
            Some(i) => *i,
            None => continue,
        };
        for dr in -1..=1 {
            for dc in -1..=1 {
                let p = (sr + dr, sc + dc);
                if let Some(pi) = point_index.get(&p) {
                    point_to_sensors.entry(*pi).or_default().push(sidx);
                }
            }
        }
    }
    (points, point_to_sensors, point_index)
}

// Petite grille : 4 x 3 = 12 points, 12 capteurs candidats
let (points_small, pts2sens_small, _pindex_small) = build_grid_cover(4, 3);
let n_small = points_small.len();
let espace_small: u128 = 1u128 << n_small;
println!("Petite grille : {} points, {} capteurs candidats", n_small, n_small);
println!("  brute-force : 2^{} = {} sous-ensembles a explorer (faisable)", n_small, espace_small);

Petite grille : 12 points, 12 capteurs candidats


  brute-force : 2^12 = 4096 sous-ensembles a explorer (faisable)


### Exemple : la petite grille

Sur cette grille de 12 points, la baseline brute-force est encore **faisable** : elle énumère les $2^{12} = 4096$ sous-ensembles de capteurs, par taille croissante, et retient le plus petit qui couvre tout. C'est notre **témoin de correction** : la valeur qu'elle donne est celle contre laquelle nous allons vérifier Z3.

In [4]:
use std::collections::{HashMap, HashSet};
use std::time::Instant;

// Cellule autonome : construction de grille et utilitaires redéfinis ici.
fn build_grid_cover(rows: i64, cols: i64) -> (Vec<(i64, i64)>, HashMap<usize, Vec<usize>>, HashMap<(i64, i64), usize>) {
    let mut points: Vec<(i64, i64)> = Vec::new();
    for r in 0..rows {
        for c in 0..cols {
            points.push((r, c));
        }
    }
    let mut point_index: HashMap<(i64, i64), usize> = HashMap::new();
    for (i, p) in points.iter().enumerate() {
        point_index.insert(*p, i);
    }
    let mut point_to_sensors: HashMap<usize, Vec<usize>> = HashMap::new();
    for (sr, sc) in points.iter().copied() {
        let sidx = match point_index.get(&(sr, sc)) {
            Some(i) => *i,
            None => continue,
        };
        for dr in -1..=1 {
            for dc in -1..=1 {
                let p = (sr + dr, sc + dc);
                if let Some(pi) = point_index.get(&p) {
                    point_to_sensors.entry(*pi).or_default().push(sidx);
                }
            }
        }
    }
    (points, point_to_sensors, point_index)
}

// Équivalent Rust de itertools.combinations : sous-ensembles de taille k
// énumérés dans l'ordre lexicographique croissant.
fn combinations(n: usize, k: usize) -> Vec<Vec<usize>> {
    let mut out: Vec<Vec<usize>> = Vec::new();
    if k > n {
        return out;
    }
    if k == 0 {
        out.push(Vec::new());
        return out;
    }
    let mut idx: Vec<usize> = (0..k).collect();
    loop {
        out.push(idx.clone());
        let mut i = k;
        loop {
            if i == 0 {
                return out;
            }
            i -= 1;
            if idx[i] != i + n - k {
                break;
            }
        }
        idx[i] += 1;
        for j in (i + 1)..k {
            idx[j] = idx[j - 1] + 1;
        }
    }
}

// Parcours par taille croissante : renvoie le plus petit sous-ensemble de
// capteurs qui couvre tous les points. Complet en 2^k au pire.
fn brute_force_set_cover(n_sensors: usize, point_to_sensors: &HashMap<usize, Vec<usize>>) -> (Vec<usize>, usize) {
    let points: Vec<usize> = point_to_sensors.keys().copied().collect();
    for k in 0..=n_sensors {
        for combo in combinations(n_sensors, k) {
            let chosen: HashSet<usize> = combo.iter().copied().collect();
            let couvre = points.iter().all(|p| match point_to_sensors.get(p) {
                Some(sensors) => sensors.iter().any(|s| chosen.contains(s)),
                None => false,
            });
            if couvre {
                return (combo, k);
            }
        }
    }
    (Vec::new(), n_sensors)
}

let (points_small, pts2sens_small, _pindex_small) = build_grid_cover(4, 3);
let n_small = points_small.len();

let t0 = Instant::now();
let (bf_selected, bf_k) = brute_force_set_cover(n_small, &pts2sens_small);
let bf_time = t0.elapsed().as_secs_f64();
println!("Brute-force : couverture minimale de {} capteurs en {:.3} s", bf_k, bf_time);
println!("  capteurs choisis (indices) : {:?}", bf_selected);

Brute-force : couverture minimale de 2 capteurs en 0.000 s


  capteurs choisis (indices) : [1, 7]


### Interpretation : la baseline brute-force

La brute-force donne la taille de la couverture minimale sur la petite grille **en énumérant** tous les sous-ensembles. C'est honnête et exact — mais son coût est $2^k$ en le nombre $k$ de capteurs candidats : ici $2^{12}$, donc instantané ; plus loin, ce sera rédhibitoire. C'est cette courbe qui va dessiner la leçon du notebook.

## 2. Le solveur SMT : Z3

**Z3** est un solveur **SMT** (*Satisfiability Modulo Théories*) : on y déclare des variables typées (booléennes, entières, réelles, tableaux, bit-vectors) et des contraintes, et il décide la satisfyingabilité — et, via son module **`Optimize`**, optimise un objectif.

La traduction du set cover en SMT d'optimisation est directe :

| Élément du problème | Encodage Z3 |
|---|---|
| Un capteur candidat | une variable booléenne $x_i$ |
| « le point $p$ est couvert » | $\bigvee_{i \in \text{capteurs}(p)} x_i$ |
| « couverture de tous les points » | une telle contrainte par point |
| « minimiser le nombre de capteurs » | `minimize(&cost)` où `cost = Σ x_i` |

Le solveur ne **cherche plus** : on **déclare** ce qu'est une couverture valide, on **demande** la plus petite, et Z3 la produit. C'est le même réflexe déclaratif que CP-SAT dans CSP-5 — seul le solveur a changé.

In [5]:
use std::collections::{HashMap, HashSet};
use z3::ast::{Ast, Bool, Int};
use z3::{Optimize, SatResult, Solver};

// Cellule autonome : toutes les fonctions utilisees sont redéfinies ici.
// La dependance z3 est declaree une seule fois, en tete de carnet (cellule 1) :
// aucune directive :dep ici.
fn build_grid_cover(rows: i64, cols: i64) -> (Vec<(i64, i64)>, HashMap<usize, Vec<usize>>, HashMap<(i64, i64), usize>) {
    let mut points = Vec::new();
    for r in 0..rows {
        for c in 0..cols {
            points.push((r, c));
        }
    }
    let mut point_index = HashMap::new();
    for (i, p) in points.iter().enumerate() {
        point_index.insert(*p, i);
    }
    let mut point_to_sensors: HashMap<usize, Vec<usize>> = HashMap::new();
    for (sr, sc) in points.iter().copied() {
        let sidx = match point_index.get(&(sr, sc)) {
            Some(i) => *i,
            None => continue,
        };
        for dr in -1..=1 {
            for dc in -1..=1 {
                let p = (sr + dr, sc + dc);
                if let Some(pi) = point_index.get(&p) {
                    point_to_sensors.entry(*pi).or_default().push(sidx);
                }
            }
        }
    }
    (points, point_to_sensors, point_index)
}

fn combinations(n: usize, k: usize) -> Vec<Vec<usize>> {
    let mut out = Vec::new();
    if k > n {
        return out;
    }
    if k == 0 {
        out.push(Vec::new());
        return out;
    }
    let mut idx: Vec<usize> = (0..k).collect();
    loop {
        out.push(idx.clone());
        let mut i = k;
        loop {
            if i == 0 {
                return out;
            }
            i -= 1;
            if idx[i] != i + n - k {
                break;
            }
        }
        idx[i] += 1;
        for j in (i + 1)..k {
            idx[j] = idx[j - 1] + 1;
        }
    }
}

fn brute_force_set_cover(n_sensors: usize, point_to_sensors: &HashMap<usize, Vec<usize>>) -> (Vec<usize>, usize) {
    let points: Vec<usize> = point_to_sensors.keys().copied().collect();
    for k in 0..=n_sensors {
        for combo in combinations(n_sensors, k) {
            let chosen: HashSet<usize> = combo.iter().copied().collect();
            if points.iter().all(|p| match point_to_sensors.get(p) {
                Some(sensors) => sensors.iter().any(|s| chosen.contains(s)),
                None => false,
            }) {
                return (combo, k);
            }
        }
    }
    (Vec::new(), n_sensors)
}

fn solve_set_cover_z3(n_sensors: usize, point_to_sensors: &HashMap<usize, Vec<usize>>) -> Option<(Vec<usize>, i64, bool)> {
    let opt = Optimize::new();
    let x: Vec<Bool> = (0..n_sensors)
        .map(|i| Bool::new_const(format!("x{i}")))
        .collect();

    for sensors in point_to_sensors.values() {
        let literals: Vec<Bool> = sensors.iter().map(|i| x[*i].clone()).collect();
        opt.assert(Bool::or(&literals));
    }

    let one = Int::from_i64(1);
    let zero = Int::from_i64(0);
    let cost: Int = x.iter().map(|xi| xi.ite(&one, &zero)).sum();
    opt.minimize(&cost);

    if opt.check(&[]) != SatResult::Sat {
        return None;
    }
    let model = match opt.get_model() {
        Some(model) => model,
        // Aucun modele optimal disponible : la fonction renvoie None.
        // Pas de unwrap()/expect() sur ce chemin : l'absence de modele est signalee
        // explicitement a l'appelant, qui decide de l'ignorer.
        None => return None,
    };

    let mut selected = Vec::new();
    for i in 0..n_sensors {
        let value = match model.eval(&x[i], true) {
            Some(value) => value,
            // Evaluation indisponible : on abandonne plutot que de forcer une valeur.
            None => return None,
        };
        match value.as_bool() {
            Some(true) => selected.push(i),
            Some(false) => {}
            None => return None,
        }
    }

    let optimum = match model.eval(&cost, true) {
        Some(value) => match value.as_i64() {
            Some(value) => value,
            None => return None,
        },
        None => return None,
    };

    // Preuve de minimalite : aucune couverture strictement plus petite n'existe.
    let proof = Solver::new();
    let y: Vec<Bool> = (0..n_sensors)
        .map(|i| Bool::new_const(format!("y{i}")))
        .collect();
    for sensors in point_to_sensors.values() {
        let literals: Vec<Bool> = sensors.iter().map(|i| y[*i].clone()).collect();
        proof.assert(Bool::or(&literals));
    }
    let proof_cost: Int = y.iter().map(|yi| yi.ite(&one, &zero)).sum();
    proof.assert(proof_cost.lt(&Int::from_i64(optimum)));
    let minimal = proof.check() == SatResult::Unsat;

    Some((selected, optimum, minimal))
}

let (points_small, pts2sens_small, _pindex_small) = build_grid_cover(4, 3);
let n_small = points_small.len();
let (_bf_selected, bf_k) = brute_force_set_cover(n_small, &pts2sens_small);

match solve_set_cover_z3(n_small, &pts2sens_small) {
    Some((z_selected, z_opt, z_minimal)) => {
        println!("Z3 : couverture minimale de {} capteurs", z_opt);
        println!("  capteurs choisis (indices) : {:?}", z_selected);
        println!("  minimalite prouvee par Z3  : {}", if z_minimal { "True" } else { "False" });
        println!("  accord avec la brute-force : {}", if z_opt == bf_k as i64 { "True" } else { "False" });
    }
    None => {
        println!("Z3 : aucun modele optimal disponible");
    }
};


Z3 : couverture minimale de 2 capteurs


  capteurs choisis (indices) : [1, 10]


  minimalite prouvee par Z3  : True


  accord avec la brute-force : True


### Interpretation : Z3 retrouve l'optimum — et le prouve

Sur la petite grille, **Z3 et la brute-force tombent d'accord** : même taille de couverture minimale. C'est le **témoin de correction** qui valide le modèle SMT — si les deux divergeaient, ce serait le signal d'un encodage faux.

La différence est dans ce que chaque méthode **apporte** : la brute-force *trouve* un optimum en énumérant ; Z3, en plus de le trouver, **prouve sa minimalité** en démontrant qu'aucune couverture strictement plus petite n'existe (le second appel `Solver` rend `unsat`). Cette preuve est le signe distinctif du solveur : elle ne dépend pas de la taille de l'espace exploré.

## 3. Le passage à l'échelle : où le solveur prend le dessus

C'est ici que la leçon se lit. Prenons une grille $10 \times 6$ : 60 points, **60 capteurs candidats**. La baseline brute-force devrait alors énumérer $2^{60} \approx 1{,}15 \times 10^{18}$ sous-ensembles — un calcul qui ne se termine pas de sitôt. Le solveur SMT, lui, ne dépend pas de cette énumération : il raisonne sur les contraintes. C'est **la** capacité distinctive que ce notebook doit rendre *visible dans la sortie*.

In [6]:
use std::collections::HashMap;
use std::time::Instant;
use z3::ast::{Ast, Bool, Int};
use z3::{Optimize, SatResult, Solver};

// Cellule autonome : construction et solveur redéfinis ici.
// La dependance z3 est declaree une seule fois, en tete de carnet (cellule 1).
fn build_grid_cover(rows: i64, cols: i64) -> (Vec<(i64, i64)>, HashMap<usize, Vec<usize>>, HashMap<(i64, i64), usize>) {
    let mut points = Vec::new();
    for r in 0..rows {
        for c in 0..cols {
            points.push((r, c));
        }
    }
    let mut point_index = HashMap::new();
    for (i, p) in points.iter().enumerate() {
        point_index.insert(*p, i);
    }
    let mut point_to_sensors: HashMap<usize, Vec<usize>> = HashMap::new();
    for (sr, sc) in points.iter().copied() {
        let sidx = match point_index.get(&(sr, sc)) {
            Some(i) => *i,
            None => continue,
        };
        for dr in -1..=1 {
            for dc in -1..=1 {
                let p = (sr + dr, sc + dc);
                if let Some(pi) = point_index.get(&p) {
                    point_to_sensors.entry(*pi).or_default().push(sidx);
                }
            }
        }
    }
    (points, point_to_sensors, point_index)
}

fn solve_set_cover_z3(n_sensors: usize, point_to_sensors: &HashMap<usize, Vec<usize>>) -> Option<(Vec<usize>, i64, bool)> {
    let opt = Optimize::new();
    let x: Vec<Bool> = (0..n_sensors)
        .map(|i| Bool::new_const(format!("x{i}")))
        .collect();
    for sensors in point_to_sensors.values() {
        let literals: Vec<Bool> = sensors.iter().map(|i| x[*i].clone()).collect();
        opt.assert(Bool::or(&literals));
    }
    let one = Int::from_i64(1);
    let zero = Int::from_i64(0);
    let cost: Int = x.iter().map(|xi| xi.ite(&one, &zero)).sum();
    opt.minimize(&cost);
    if opt.check(&[]) != SatResult::Sat {
        return None;
    }
    let model = match opt.get_model() {
        Some(model) => model,
        // Aucun modele optimal disponible : la fonction renvoie None.
        // Pas de unwrap()/expect() sur ce chemin : l'absence de modele est signalee
        // explicitement a l'appelant.
        None => return None,
    };
    let mut selected = Vec::new();
    for i in 0..n_sensors {
        let value = match model.eval(&x[i], true) {
            Some(value) => value,
            // Evaluation indisponible : on abandonne plutot que de forcer une valeur.
            None => return None,
        };
        match value.as_bool() {
            Some(true) => selected.push(i),
            Some(false) => {}
            None => return None,
        }
    }
    let optimum = match model.eval(&cost, true) {
        Some(value) => match value.as_i64() {
            Some(value) => value,
            None => return None,
        },
        None => return None,
    };
    let proof = Solver::new();
    let y: Vec<Bool> = (0..n_sensors)
        .map(|i| Bool::new_const(format!("y{i}")))
        .collect();
    for sensors in point_to_sensors.values() {
        let literals: Vec<Bool> = sensors.iter().map(|i| y[*i].clone()).collect();
        proof.assert(Bool::or(&literals));
    }
    let proof_cost: Int = y.iter().map(|yi| yi.ite(&one, &zero)).sum();
    proof.assert(proof_cost.lt(&Int::from_i64(optimum)));
    let minimal = proof.check() == SatResult::Unsat;
    Some((selected, optimum, minimal))
}

// Grande grille : 10 x 6 = 60 points, 60 capteurs candidats
let (points_large, pts2sens_large, _pindex_large) = build_grid_cover(10, 6);
let n_large = points_large.len();
let espace_large: u128 = 1u128 << n_large;
println!("Grande grille : {} points, {} capteurs candidats", n_large, n_large);
println!("  brute-force : 2^{} = {} sous-ensembles  -> infeasible", n_large, espace_large);
println!();

let t0 = Instant::now();
match solve_set_cover_z3(n_large, &pts2sens_large) {
    Some((_lz_selected, lz_opt, lz_minimal)) => {
        let lz_time = t0.elapsed().as_secs_f64();
        println!("Z3 : couverture minimale de {} capteurs en {:.1} ms", lz_opt, lz_time * 1000.0);
        println!("  minimalite prouvee par Z3  : {}", if lz_minimal { "True" } else { "False" });
    }
    None => {
        let lz_time = t0.elapsed().as_secs_f64();
        println!("Z3 : aucun modele optimal disponible en {:.1} ms", lz_time * 1000.0);
    }
};


Grande grille : 60 points, 60 capteurs candidats


  brute-force : 2^60 = 1152921504606846976 sous-ensembles  -> infeasible


Z3 : couverture minimale de 8 capteurs en 45.3 ms


  minimalite prouvee par Z3  : True


In [7]:
use std::collections::{HashMap, HashSet};
use z3::ast::{Ast, Bool, Int};
use z3::{Optimize, SatResult, Solver};

// Cellule autonome : construction et solveur redéfinis ici afin de ne
// dépendre d'aucune cellule précédente sous evcxr.
// La dependance z3 est declaree une seule fois, en tete de carnet (cellule 1).
fn build_grid_cover(rows: i64, cols: i64) -> (Vec<(i64, i64)>, HashMap<usize, Vec<usize>>, HashMap<(i64, i64), usize>) {
    let mut points = Vec::new();
    for r in 0..rows {
        for c in 0..cols {
            points.push((r, c));
        }
    }
    let mut point_index = HashMap::new();
    for (i, p) in points.iter().enumerate() {
        point_index.insert(*p, i);
    }
    let mut point_to_sensors: HashMap<usize, Vec<usize>> = HashMap::new();
    for (sr, sc) in points.iter().copied() {
        let sidx = match point_index.get(&(sr, sc)) {
            Some(i) => *i,
            None => continue,
        };
        for dr in -1..=1 {
            for dc in -1..=1 {
                let p = (sr + dr, sc + dc);
                if let Some(pi) = point_index.get(&p) {
                    point_to_sensors.entry(*pi).or_default().push(sidx);
                }
            }
        }
    }
    (points, point_to_sensors, point_index)
}

fn solve_set_cover_z3(n_sensors: usize, point_to_sensors: &HashMap<usize, Vec<usize>>) -> Option<(Vec<usize>, i64, bool)> {
    let opt = Optimize::new();
    let x: Vec<Bool> = (0..n_sensors)
        .map(|i| Bool::new_const(format!("x{i}")))
        .collect();
    for sensors in point_to_sensors.values() {
        let literals: Vec<Bool> = sensors.iter().map(|i| x[*i].clone()).collect();
        opt.assert(Bool::or(&literals));
    }
    let one = Int::from_i64(1);
    let zero = Int::from_i64(0);
    let cost: Int = x.iter().map(|xi| xi.ite(&one, &zero)).sum();
    opt.minimize(&cost);
    if opt.check(&[]) != SatResult::Sat {
        return None;
    }
    let model = match opt.get_model() {
        Some(model) => model,
        // Aucun modele optimal disponible : la fonction renvoie None.
        // Pas de unwrap()/expect() sur ce chemin : l'absence de modele est signalee
        // explicitement a l'appelant.
        None => return None,
    };
    let mut selected = Vec::new();
    for i in 0..n_sensors {
        let value = match model.eval(&x[i], true) {
            Some(value) => value,
            // Evaluation indisponible : on abandonne plutot que de forcer une valeur.
            None => return None,
        };
        match value.as_bool() {
            Some(true) => selected.push(i),
            Some(false) => {}
            None => return None,
        }
    }
    let optimum = match model.eval(&cost, true) {
        Some(value) => match value.as_i64() {
            Some(value) => value,
            None => return None,
        },
        None => return None,
    };
    let proof = Solver::new();
    let y: Vec<Bool> = (0..n_sensors)
        .map(|i| Bool::new_const(format!("y{i}")))
        .collect();
    for sensors in point_to_sensors.values() {
        let literals: Vec<Bool> = sensors.iter().map(|i| y[*i].clone()).collect();
        proof.assert(Bool::or(&literals));
    }
    let proof_cost: Int = y.iter().map(|yi| yi.ite(&one, &zero)).sum();
    proof.assert(proof_cost.lt(&Int::from_i64(optimum)));
    let minimal = proof.check() == SatResult::Unsat;
    Some((selected, optimum, minimal))
}

// Rendu textuel de la solution : '.' = point, '*' = capteur choisi,
// '+' = point couvert. Cette version ne produit aucune figure.
fn plot_grid_cover(points: &[(i64, i64)], point_index: &HashMap<(i64, i64), usize>, selected: &[usize], title: &str) -> String {
    let rows = points.iter().map(|(r, _)| *r).max().unwrap_or(0) + 1;
    let cols = points.iter().map(|(_, c)| *c).max().unwrap_or(0) + 1;
    let mut covered: HashSet<(i64, i64)> = HashSet::new();
    for sensor in selected {
        if let Some((sr, sc)) = points.get(*sensor).copied() {
            for dr in -1..=1 {
                for dc in -1..=1 {
                    let p = (sr + dr, sc + dc);
                    if point_index.contains_key(&p) {
                        covered.insert(p);
                    }
                }
            }
        }
    }
    let mut rendu = String::new();
    rendu.push_str(title);
    rendu.push('\n');
    for r in 0..rows {
        for c in 0..cols {
            let p = (r, c);
            let character = if selected.iter().any(|i| points.get(*i) == Some(&p)) {
                '*'
            } else if covered.contains(&p) {
                '+'
            } else {
                '.'
            };
            rendu.push(character);
        }
        rendu.push('\n');
    }
    rendu
}

let (points_large, pts2sens_large, pindex_large) = build_grid_cover(10, 6);
let n_large = points_large.len();
match solve_set_cover_z3(n_large, &pts2sens_large) {
    Some((lz_selected, lz_opt, _lz_minimal)) => {
        let rendu = plot_grid_cover(
            &points_large,
            &pindex_large,
            &lz_selected,
            &format!("Grande grille : {} capteurs couvrent les {} points", lz_opt, points_large.len()),
        );
        println!("{}", rendu);
    }
    None => {
        println!("Z3 : aucun modele optimal disponible");
    }
};

Grande grille : 8 capteurs couvrent les 60 points


+*++++


++++*+


++++++


+*++++


++++*+


++++++


+*++*+


++++++


++++++


+*++*+


### Interpretation : le passage à l'échelle

Là où la brute-force s'effondre ($2^{60} \approx 10^{18}$ sous-ensembles, inenvisageable), **Z3 renvoie l'optimum en quelques millisecondes** — et, comme sur la petite grille, **prouve sa minimalité**. Le rendu textuel montre la solution : un maillage clair de capteurs qui couvre la grille entière au nombre minimal. C'est la capacité distinctive du solveur SMT, rendue tangible : pas une heuristique approchante, un **optimum certifié** sur un problème NP-complet de taille réelle.

## 4. Exercices

Les trois exercices ci-dessous prolongent le modèle vu plus haut : une contrainte de budget, une hétérogénéité de portée, et une preuve de borne inférieure. Les fonctions `build_grid_cover` et `solve_set_cover_z3` sont réutilisables telles quelles ; sous evcxr, elles doivent toutefois être redéfinies dans chaque cellule de code autonome.

### Exercice 1 : couverture sous contrainte de budget

**Énoncé** : Sur la grande grille, le budget ne permet d'allumer que **K capteurs**. Modélisez en remplaçant l'objectif `opt.minimize(&cost)` (où `cost = Σ x[i]`) par la **contrainte** `Σ x[i] ≤ K`, puis déterminez la **plus petite** valeur de `K` pour laquelle tous les points restent couverts.

**Consignes** :
1. Réutilisez le modèle de `solve_set_cover_z3`
2. Remplacez l'objectif par la contrainte de budget `Σ x[i] ≤ K`
3. Trouvez le plus petit `K` admissible (indice : il coïncide avec l'optimum du `minimize` — vérifiez en testant la satisfaisabilité pour des `K` décroissants)

In [8]:
// Exercice 1 : couverture sous contrainte de budget
// Indice : le plus petit K admissible est exactement l'optimum renvoye par le minimize.
// Construire un Optimize (ou Solver) avec les contraintes de couverture et
// l'addition Sum(x) <= K, puis faire check() pour des K decroissants depuis l'optimum.

// Votre code ici
let budget_minimal: Option<i64> = None;
println!("Exercice a completer");

Exercice a completer


### Exercice 2 : capteurs à portée différente

**Énoncé** : Certains capteurs couvrent un **carré 3×3**, d'autres une **croix** (5 cases : le centre et ses 4 voisins directs). Mélangez les deux types sur la **petite** grille et re-déterminez la couverture minimale.

**Consignes** :
1. Étendez `build_grid_cover` pour accepter un **rayon / une forme** par capteur
2. Re-solvez avec `solve_set_cover_z3` sur le nouveau `point_to_sensors`
3. Comparez l'optimum obtenu à celui du carré 3×3 seul

In [9]:
// Exercice 2 : capteurs a portee differente (carre 3x3 vs croix 5 cases)
// Indice : une croix couvre {(r,c), (r±1,c), (r,c±1)} ; construisez le
// point_to_sensors correspondant puis reutilisez solve_set_cover_z3.

// Votre code ici
let point_to_sensors_mixte: Option<std::collections::HashMap<usize, Vec<usize>>> = None;
println!("Exercice a completer");

Exercice a completer


### Exercice 3 : prouver qu'on ne peut pas faire mieux

**Énoncé** : Z3 renvoie un optimum `m` et « prouve la minimalité » en montrant qu'aucune couverture de taille `< m` n'existe. Reformulez cette preuve vous-même avec un **`Solver` pur** (sans `Optimize`) : montrez que le modèle restreint à `Σ x[i] ≤ m − 1` est **insatisfaisable**.

**Consignes** :
1. Construisez un `Solver::new()` avec les seules contraintes de couverture
2. Ajoutez `Σ x[i] ≤ m − 1`
3. Vérifiez que le statut est `unsat` et interprétez ce que cela prouve

In [10]:
// Exercice 3 : prouver qu'on ne peut pas faire mieux
// Indice : c'est exactement le bloc « Preuve de minimalite » de solve_set_cover_z3,
// a reformuler avec un Solver pur et add(Sum(x) <= m - 1), puis check() == unsat.

// Votre code ici
let minimalite_prouvee: Option<bool> = None;
println!("Exercice a completer");

Exercice a completer


## Références

1. **Z3** — de Moura, L. & Bjørner, N. (2008). « Z3: An Efficient SMT Solver ». *Tools and Algorithms for the Construction and Analysis of Systems (TACAS)*, LNCS 4956:357-371. Le solveur SMT utilisé ici (crate Rust `z3` 0.24.0).
2. **Set Cover** — Garey, M.R. & Johnson, D.S. (1979). *Computers and Intractability: A Guide to the Theory of NP-Completeness*. W.H. Freeman. La couverture d'ensembles est l'un des 21 problèmes NP-complets fondateurs.
3. **Le pont CP ↔ SAT ↔ SMT** — Bardin, S., Bjørner, N. & Cadar, C. (2019). *Bringing CP, SAT and SMT together: Next Challenges in Constraint Solving*. Dagstuhl Reports 9(2):27. — [doi:10.4230/DagRep.9.2.27](https://doi.org/10.4230/DagRep.9.2.27). La référence qui fonde ce notebook : le même problème déclaratif attaqué par trois familles de solveurs.
4. **Ensemble dominant** — le cas grille de ce notebook est un cas particulier de l'ensemble dominant d'un graphe, lui-même NP-complet (cf. [2]).

## À retenir

Ce notebook a franchi la frontière CP → SMT qui se dessinait depuis CSP-6 (Lazy Clause Generation) : le même problème de **couverture minimale** — NP-complet, central en placement de capteurs et en logistique — a été formulé une seule fois, puis résolu par deux familles de solveurs.

1. **L'énumération exhaustive est hors de portée dès que l'instance grossit** : elle explore $2^k$ sous-ensembles, soit 4096 pour la petite grille (12 points, 12 capteurs) et 1 152 921 504 606 846 976 pour la grande grille (60 points).
2. **Sur la petite grille, la force brute fournit le témoin de correction** : elle trouve une couverture minimale de **2** capteurs (indices [1, 7]) ; Z3 aboutit à la même taille et l'accord des deux voies est vérifié. Les indices retenus par Z3 peuvent différer : la couverture minimale n'est pas unique, seule la taille est comparée.
3. **Z3 ne se contente pas de trouver un optimum, il le prouve** : la minimalité est obtenue en montrant insatisfaisable le modèle restreint à $\sum x_i \le m - 1$ — c'est-à-dire en refusant explicitement toute couverture plus petite.
4. **À l'échelle, l'écart devient rédhibitoire** : l'énumération s'effondre, et Z3 renvoie une couverture minimale de **8** capteurs avec minimalité prouvée, en quelques dizaines de millisecondes.
5. **La promesse du paradigme déclaratif est tenue** : on ne change pas de *discipline* en changeant de *solveur* — la formulation, elle, reste.

## Vérifiez votre compréhension

1. Pourquoi Z3 et la force brute ne choisissent-elles pas les mêmes indices de capteurs alors que la taille minimale est la même ?
2. Comment Z3 prouve-t-il qu'aucune couverture de 7 capteurs n'existe sur la grande grille ?
3. Quel est l'intérêt de conserver une baseline brute-force si Z3 fait mieux ?

<details><summary>Réponses</summary>

1. Parce que la couverture minimale n'est pas unique : plusieurs paires de capteurs atteignent la taille 2, et les deux méthodes explorent l'espace dans des ordres différents. Ce qui est vérifié — et qui compte — est l'égalité de la taille minimale, pas l'égalité du choix.
2. En résolvant le modèle restreint à $\sum x_i \le 7$ : il est insatisfaisable, donc aucune couverture de 7 capteurs n'existe, donc 8 est minimal. C'est une preuve, là où la force brute se contente d'une énumération.
3. Elle fournit un témoin de correction indépendant : sans elle, on ne saurait pas que l'optimum de Z3 est bien l'optimum. Sur la petite grille, les deux voies concordent ; ce désaccord-là, il est vérifiable.

</details>

## Pour aller plus loin

- **Carnets voisins** : [CSP-06-Hybridization-Python.ipynb](CSP-06-Hybridization-Python.ipynb) — la Lazy Clause Generation, d'où le passage CP → SAT → SMT ; [CSP-05-Optimization-Python.ipynb](CSP-05-Optimization-Python.ipynb) — la même famille de problèmes d'optimum, résolue sous CP-SAT.
- **Lectures** : la section « Références » en bas de carnet récapitule les quatre notices — le solveur SMT utilisé ici, la NP-complétude de la couverture et de l'ensemble dominant, et le passage d'une famille de solveurs à l'autre.
